In [1]:
import numpy as np
from typing import List, Optional, Tuple, Dict, Any

In [2]:
def gmres(A, b, x0=None, tol=1e-8, maxiter=None, restart=None):

    n = b.shape[0]
    if x0 is None:
        x = np.zeros_like(b, dtype=float)
    else:
        x = np.array(x0, dtype=float).copy()

    if maxiter is None:
        maxiter = n

    def matvec(v):
        return A(v) if callable(A) else A @ v

    b = np.array(b, dtype=float)
    restart = n if restart is None else int(restart)

    total_iters = 0
    history = []

    r = b - matvec(x)
    beta = np.linalg.norm(r)
    if beta == 0.0:
        return x, {"converged": True, "iterations": 0, "residual_norm": 0.0, "history": [0.0]}

    while total_iters < maxiter:
        m = min(restart, maxiter - total_iters)
        V = np.zeros((n, m + 1), dtype=float)
        H = np.zeros((m + 1, m), dtype=float)
        cs = np.zeros(m, dtype=float)
        sn = np.zeros(m, dtype=float)
        g = np.zeros(m + 1, dtype=float)

        r = b - matvec(x)
        beta = np.linalg.norm(r)
        if beta == 0.0:
            return x, {"converged": True, "iterations": total_iters, "residual_norm": 0.0, "history": history}

        V[:, 0] = r / beta
        g[0] = beta

        inner_converged = False
        for j in range(m):
            total_iters += 1

            w = matvec(V[:, j])

            for i in range(j + 1):
                H[i, j] = np.dot(V[:, i], w)
                w = w - H[i, j] * V[:, i]

            H[j + 1, j] = np.linalg.norm(w)
            if H[j + 1, j] != 0 and j + 1 < m + 1:
                V[:, j + 1] = w / H[j + 1, j]

            for i in range(j):
                temp = cs[i] * H[i, j] + sn[i] * H[i + 1, j]
                H[i + 1, j] = -sn[i] * H[i, j] + cs[i] * H[i + 1, j]
                H[i, j] = temp

            denom = np.hypot(H[j, j], H[j + 1, j])
            if denom == 0.0:
                cs[j] = 1.0
                sn[j] = 0.0
            else:
                cs[j] = H[j, j] / denom
                sn[j] = H[j + 1, j] / denom

            H[j, j] = cs[j] * H[j, j] + sn[j] * H[j + 1, j]
            H[j + 1, j] = 0.0

            g_j = g[j]
            g[j] = cs[j] * g_j
            g[j + 1] = -sn[j] * g_j

            resid = abs(g[j + 1])
            history.append(resid)

            if resid < tol:
                y = np.linalg.solve(H[:j + 1, :j + 1], g[:j + 1])
                x = x + V[:, :j + 1] @ y
                inner_converged = True
                break

        if not inner_converged:
            y = np.linalg.lstsq(H[:m, :m], g[:m], rcond=None)[0]
            x = x + V[:, :m] @ y
            r = b - matvec(x)
            if np.linalg.norm(r) < tol:
                return x, {"converged": True, "iterations": total_iters, "residual_norm": float(np.linalg.norm(r)), "history": history}

        if total_iters >= maxiter:
            break

    final_resid = float(np.linalg.norm(b - matvec(x)))
    return x, {"converged": final_resid < tol, "iterations": total_iters, "residual_norm": final_resid, "history": history}

In [3]:
def mod_inv(a: int, p: int) -> int:
    a %= p
    if a == 0:
        raise ZeroDivisionError("No inverse for 0 modulo p")
    return pow(a, -1, p)


def mod_matvec(A: np.ndarray, x: np.ndarray, p: int) -> np.ndarray:
    return (np.array(A, dtype=int) @ np.array(x, dtype=int)) % p


def mod_dot(u: np.ndarray, v: np.ndarray, p: int) -> int:
    return int(np.dot(np.array(u, dtype=int), np.array(v, dtype=int)) % p)


def _rref_matrix_mod_p(M: np.ndarray, p: int):
    A = np.array(M, dtype=int) % p
    rows, cols = A.shape
    row = 0
    pivots = []

    for col in range(cols):
        pivot = None
        for r in range(row, rows):
            if A[r, col] % p != 0:
                pivot = r
                break
        if pivot is None:
            continue

        if pivot != row:
            A[[row, pivot]] = A[[pivot, row]]

        inv = mod_inv(int(A[row, col]), p)
        A[row, :] = (A[row, :] * inv) % p

        for r in range(rows):
            if r != row and A[r, col] % p != 0:
                factor = A[r, col] % p
                A[r, :] = (A[r, :] - factor * A[row, :]) % p

        pivots.append(col)
        row += 1
        if row == rows:
            break

    return A % p, pivots


def rref_mod_p(A: np.ndarray, b: np.ndarray, p: int):
    A = np.array(A, dtype=int) % p
    b = np.array(b, dtype=int).reshape(-1, 1) % p
    n, m = A.shape
    aug = np.hstack([A, b]).astype(int) % p
    rref, pivots = _rref_matrix_mod_p(aug, p)

    rank = len(pivots)
    for r in range(rank, n):
        if np.all(rref[r, :m] % p == 0) and rref[r, m] % p != 0:
            return None, False, rank

    x = np.zeros(m, dtype=int)
    for i, col in enumerate(pivots):
        x[col] = rref[i, m] % p

    return x % p, True, rank


def nullspace_mod_p(M: np.ndarray, p: int) -> List[np.ndarray]:
    A = np.array(M, dtype=int) % p
    rref, pivots = _rref_matrix_mod_p(A, p)
    rows, cols = A.shape
    pivot_set = set(pivots)
    free_cols = [j for j in range(cols) if j not in pivot_set]

    basis = []
    for free in free_cols:
        x = np.zeros(cols, dtype=int)
        x[free] = 1
        for i in reversed(range(len(pivots))):
            col = pivots[i]
            s = 0
            for j in range(col + 1, cols):
                if rref[i, j] % p != 0:
                    s = (s + rref[i, j] * x[j]) % p
            x[col] = (-s) % p
        basis.append(x % p)

    return basis


def berlekamp_massey(sequence: List[int], p: int) -> List[int]:
    seq = [int(x) % p for x in sequence]
    C = [1]
    B = [1]
    L = 0
    m = 1
    b = 1

    for n in range(len(seq)):
        d = seq[n]
        for i in range(1, L + 1):
            d = (d + C[i] * seq[n - i]) % p
        if d == 0:
            m += 1
            continue

        T = C[:]
        coef = d * mod_inv(b, p) % p
        if len(C) < len(B) + m:
            C += [0] * (len(B) + m - len(C))
        for i in range(len(B)):
            C[i + m] = (C[i + m] - coef * B[i]) % p

        if 2 * L <= n:
            L_new = n + 1 - L
            B = T
            L = L_new
            b = d
            m = 1
        else:
            m += 1

    return [c % p for c in C[:L + 1]]


def krylov_scalar_sequence(A: np.ndarray, b: np.ndarray, u: np.ndarray, steps: int, p: int) -> List[int]:
    A = np.array(A, dtype=int) % p
    v = np.array(b, dtype=int) % p
    u = np.array(u, dtype=int) % p
    seq = []
    for _ in range(steps):
        seq.append(mod_dot(u, v, p))
        v = (A @ v) % p
    return seq


def random_vector(n: int, p: int, rng: np.random.Generator) -> np.ndarray:
    return rng.integers(0, p, size=n, dtype=int)


def polynomial_solution_from_recurrence(A: np.ndarray, b: np.ndarray, C: List[int], p: int) -> np.ndarray:
    A = np.array(A, dtype=int) % p
    v = np.array(b, dtype=int) % p
    x = np.zeros_like(v, dtype=int)
    for coeff in C[1:]:
        x = (x - int(coeff) * v) % p
        v = (A @ v) % p
    return x % p


def wiedemann_solve(
    A: np.ndarray,
    b: np.ndarray,
    p: int,
    *,
    max_retries: int = 20,
    steps: Optional[int] = None,
    seed: Optional[int] = None,
    validate: bool = True,
) -> Tuple[np.ndarray, Dict[str, Any]]:

    A = np.array(A, dtype=int) % p
    b = np.array(b, dtype=int) % p
    n, m = A.shape
    if n != m:
        raise ValueError("Wiedemann solver requires a square matrix")

    rng = np.random.default_rng(seed)
    if np.all(b % p == 0):
        zero = np.zeros(n, dtype=int)
        return zero, {
            "converged": True,
            "iterations": 0,
            "attempts": 0,
            "residual_norm": 0,
            "method": "wiedemann_krylov",
            "history": [],
        }

    history = []

    for attempt in range(1, max_retries + 1):
        vectors = [b.copy()]
        for _ in range(n):
            vectors.append((A @ vectors[-1]) % p)

        if steps is not None:
            chain = vectors[: max(2, min(len(vectors), steps + 1))]
        else:
            chain = vectors

        found = False
        for _trial in range(min(5, n + 1)):
            u = random_vector(n, p, rng)
            seq = krylov_scalar_sequence(A, b, u, len(chain), p)
            C = berlekamp_massey(seq, p)
            if len(C) <= 1 or C[0] % p == 0:
                continue
            if C[0] % p != 0:
                x = polynomial_solution_from_recurrence(A, b, C, p)
                residual = (A @ x - b) % p
                ok = bool(np.all(residual == 0))
                history.append({
                    "attempt": attempt,
                    "trial": _trial + 1,
                    "projection": u.tolist(),
                    "sequence": seq,
                    "relation": C.copy(),
                    "solution_ok": ok,
                })
                if ok:
                    found = True
                    return x % p, {
                        "converged": True,
                        "iterations": len(C) - 1,
                        "attempts": attempt,
                        "residual_norm": 0,
                        "method": "wiedemann_krylov",
                        "relation": C,
                        "history": history,
                    }

        mat = np.column_stack(chain)
        null_basis = nullspace_mod_p(mat, p)
        candidate_relation = None
        for rel in null_basis:
            if rel[0] % p != 0:
                candidate_relation = rel % p
                break

        if candidate_relation is None:
            history.append({"attempt": attempt, "status": "no_relation_with_nonzero_constant_term"})
            continue

        c0 = int(candidate_relation[0] % p)
        inv_c0 = mod_inv(c0, p)
        x = np.zeros(n, dtype=int)
        power = b.copy()
        for coeff in candidate_relation[1:]:
            x = (x - int(coeff) * power) % p
            power = (A @ power) % p
        x = (inv_c0 * x) % p

        residual = (A @ x - b) % p
        ok = bool(np.all(residual == 0))
        history.append({
            "attempt": attempt,
            "relation": candidate_relation.copy(),
            "solution_ok": ok,
        })

        if ok:
            return x % p, {
                "converged": True,
                "iterations": len(candidate_relation) - 1,
                "attempts": attempt,
                "residual_norm": 0,
                "method": "wiedemann_krylov",
                "relation": candidate_relation,
                "history": history,
            }

    if validate:
        x_fallback, ok, rank = rref_mod_p(A, b, p)
        if ok:
            return x_fallback % p, {
                "converged": True,
                "iterations": None,
                "attempts": max_retries,
                "residual_norm": 0,
                "method": "gauss_jordan_fallback",
                "rank": rank,
                "history": history,
            }

    raise RuntimeError(
        "Wiedemann-style Krylov solver did not converge after retries; the matrix may be singular "
        "or the chosen Krylov chain was not sufficient."
    )

In [4]:
def demo_real_gmres():
    np.random.seed(1)
    n = 8
    A = np.random.randn(n, n)
    A += 2.5 * np.eye(n)
    x_true = np.random.randn(n)
    b = A @ x_true
    x, info = gmres(A, b, tol=1e-10)
    print("GMRES over R")
    print("converged:", info["converged"])
    print("iterations:", info["iterations"])
    print("residual norm:", info["residual_norm"])
    print("error norm:", np.linalg.norm(x - x_true))


def demo_mod_p_solver():
    p = 11
    A = np.array([
        [1, 2, 0, 4],
        [0, 3, 5, 6],
        [2, 0, 1, 1],
        [4, 1, 0, 2],
    ], dtype=int)
    x_true = np.array([7, 3, 9, 1], dtype=int)
    b = (A @ x_true) % p
    x, ok, rank = rref_mod_p(A, b, p)
    print("\nExact solve over GF(p)")
    print("p =", p)
    print("ok:", ok, "rank:", rank)
    print("x recovered:", x)
    print("check Ax=b:", np.all((A @ x) % p == b))


def demo_wiedemann():
    p = 11
    np.random.seed(4)
    A = np.array([
        [4, 1, 0, 2, 3],
        [1, 5, 2, 0, 1],
        [0, 3, 6, 1, 0],
        [2, 0, 1, 4, 1],
        [1, 2, 0, 1, 7],
    ], dtype=int) % p
    x_true = np.array([2, 9, 4, 6, 1], dtype=int)
    b = (A @ x_true) % p

    x, info = wiedemann_solve(A, b, p, max_retries=30, seed=123)
    print("\nWiedemann / Krylov analogue over GF(p)")
    print("method:", info["method"])
    print("converged:", info["converged"])
    print("attempts:", info["attempts"])
    print("x recovered:", x)
    print("matches true x:", np.all(x % p == x_true % p))
    print("check Ax=b:", np.all((A @ x) % p == b))

    u = np.array([1, 2, 3, 4, 5], dtype=int)
    seq = krylov_scalar_sequence(A, b, u, 10, p)
    C = berlekamp_massey(seq, p)
    print("sequence:", seq)
    print("BM polynomial:", C)


demo_real_gmres()
demo_mod_p_solver()
demo_wiedemann()

GMRES over R
converged: True
iterations: 8
residual norm: 3.263375893225244e-15
error norm: 2.8539696776249214e-15

Exact solve over GF(p)
p = 11
ok: True rank: 4
x recovered: [7 3 9 1]
check Ax=b: True

Wiedemann / Krylov analogue over GF(p)
method: wiedemann_krylov
converged: True
attempts: 1
x recovered: [2 9 4 6 1]
matches true x: True
check Ax=b: True
sequence: [7, 8, 1, 9, 8, 9, 7, 10, 0, 3]
BM polynomial: [1, 8, 4, 7, 2]
